# Glia RAG Cache — Qdrant + Redis Demo

**Start both databases first:**
```bash
docker compose up -d
```
This brings up:
- **Redis Stack** on `localhost:6379` (semantic cache) + RedisInsight UI on `:8001`
- **Qdrant** on `localhost:6333` (vector DB) + gRPC on `:6334`

**Install dependencies:**
```bash
pip install glia sentence-transformers redis redisvl groq numpy qdrant-client
```

### What this notebook covers
| Section | What you'll see |
|---|---|
| 1 | Imports & config |
| 2 | Qdrant Vector DB wrapper (real Docker container) |
| 3 | Upload documents → embed → store in Qdrant |
| 4 | Init GliaManager + wire Qdrant → Redis cache |
| 5 | Ask questions — clear CACHE HIT / MISS banner |
| 6 | Update a document in Qdrant → auto-invalidate cache |
| 7 | Inspect Redis cache contents |
| 8 | Full cleanup |

## 1. Imports & Configuration

In [84]:
import os
import json
import uuid
import time
import warnings
import numpy as np
from datetime import datetime
from copy import deepcopy
from typing import List, Dict, Any, Optional

warnings.filterwarnings("ignore")

from groq import Groq
from sentence_transformers import SentenceTransformer

from glia.manager import GliaManager
from glia.invalidator import CacheInvalidator

In [ ]:
# ── Connection & model config ────────────────────────────────────────
REDIS_URL    = "redis://localhost:6379"
INDEX_NAME   = "rag_cache"
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")
GROQ_MODEL   = "openai/gpt-oss-20b"

# ── Qdrant config ────────────────────────────────────────────────────
QDRANT_HOST  = "localhost"
QDRANT_PORT  = 6333

print(f"Redis  : {REDIS_URL}")
print(f"Index  : {INDEX_NAME}")
print(f"Model  : {GROQ_MODEL}")
print(f"Qdrant : http://{QDRANT_HOST}:{QDRANT_PORT}")

Redis  : redis://localhost:6379
Index  : rag_cache
Model  : openai/gpt-oss-20b
Qdrant : http://localhost:6333


## 2. Qdrant Vector Database Wrapper

A thin wrapper around the official `qdrant-client` that exposes the same
interface as the previous `InMemoryVectorDB` — so **every cell from Section 3
onwards is unchanged**.

Qdrant stores each document as a **point**:
```
{
  id       : str  (UUID derived from doc_id)   ← stable, deterministic
  vector   : List[float]                        ← sentence-transformer embedding
  payload  : {
    doc_id     : str,       ← matches source_id TAG in Redis cache
    text       : str,
    updated_at : str,       ← ISO-8601 timestamp used for polling watermark
    metadata   : dict,
  }
}
```

The wrapper exposes:
| Method | Used by |
|---|---|
| `upsert(collection, doc)` | Section 3 upload & Section 6 update |
| `get(collection, doc_id)` | Section 6 show-before |
| `delete(collection, doc_id)` | optional hard delete |
| `list_all(collection)` | Section 7 inspect |
| `search(collection, query_vec, top_k)` | `ask()` retrieval |
| `describe_collection(collection)` | liveness probe (`VectorDBAdapter`) |
| `fetch_updated(collection, since, ...)` | `VectorDBAdapter` polling |


In [86]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    Filter,
    FieldCondition,
    MatchValue,
    PayloadSchemaType,
)


class QdrantVectorDB:
    """
    Thin wrapper around qdrant-client that presents the same interface
    as the previous InMemoryVectorDB.  Drop-in replacement — nothing
    in Section 3 onward needs to change.

    doc_id → Qdrant point id mapping
    ─────────────────────────────────
    Qdrant point IDs must be unsigned integers or UUIDs.  We derive a
    deterministic UUID-v5 from each doc_id string so the mapping is
    stable across restarts (same doc_id always maps to the same point).
    """

    def __init__(self, host: str = "localhost", port: int = 6333):
        self._client = QdrantClient(host=host, port=port)
        # collection_name → vector dimension (cached after first create)
        self._dims: Dict[str, int] = {}

    # ── Internal helpers ────────────────────────────────────────────────

    @staticmethod
    def _point_id(doc_id: str) -> str:
        """Deterministic UUID-v5 from an arbitrary doc_id string."""
        return str(uuid.uuid5(uuid.NAMESPACE_DNS, doc_id))

    def _ensure_collection(self, collection: str, dim: int) -> None:
        """Create the Qdrant collection if it does not already exist."""
        existing = [c.name for c in self._client.get_collections().collections]
        if collection not in existing:
            self._client.create_collection(
                collection_name=collection,
                vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
            )
            # Create a payload index on doc_id for fast exact-match lookups
            self._client.create_payload_index(
                collection_name=collection,
                field_name="doc_id",
                field_schema=PayloadSchemaType.KEYWORD,
            )
            print(f"  Created Qdrant collection '{collection}' (dim={dim})")
        self._dims[collection] = dim

    # ── CRUD ────────────────────────────────────────────────────────────

    def upsert(self, collection: str, doc: Dict) -> None:
        """Insert or overwrite a document by doc_id (upsert semantics)."""
        embedding = doc["embedding"]
        self._ensure_collection(collection, len(embedding))

        now = datetime.utcnow()
        payload = {
            "doc_id":     doc["doc_id"],
            "text":       doc["text"],
            "updated_at": now.isoformat(),
            "metadata":   doc.get("metadata", {}),
        }
        # Store updated_at on the doc dict too so callers can read it back
        doc["updated_at"] = now

        self._client.upsert(
            collection_name=collection,
            points=[
                PointStruct(
                    id=self._point_id(doc["doc_id"]),
                    vector=embedding,
                    payload=payload,
                )
            ],
        )

    def get(self, collection: str, doc_id: str) -> Optional[Dict]:
        """Return a document dict by doc_id, or None."""
        results = self._client.scroll(
            collection_name=collection,
            scroll_filter=Filter(
                must=[FieldCondition(key="doc_id", match=MatchValue(value=doc_id))]
            ),
            limit=1,
            with_vectors=True,
            with_payload=True,
        )
        points = results[0]
        if not points:
            return None
        p = points[0]
        payload = p.payload or {}
        return {
            "doc_id":     payload.get("doc_id", doc_id),
            "text":       payload.get("text", ""),
            "embedding":  list(p.vector) if p.vector else [],
            "updated_at": datetime.fromisoformat(payload["updated_at"]) if payload.get("updated_at") else None,
            "metadata":   payload.get("metadata", {}),
        }

    def delete(self, collection: str, doc_id: str) -> bool:
        """Delete a point by doc_id; return True if it existed."""
        existing = self.get(collection, doc_id)
        if existing is None:
            return False
        self._client.delete(
            collection_name=collection,
            points_selector=Filter(
                must=[FieldCondition(key="doc_id", match=MatchValue(value=doc_id))]
            ),
        )
        return True

    def list_all(self, collection: str) -> List[Dict]:
        """Return all documents in a collection."""
        all_points = []
        offset = None
        while True:
            batch, next_offset = self._client.scroll(
                collection_name=collection,
                limit=100,
                offset=offset,
                with_vectors=True,
                with_payload=True,
            )
            all_points.extend(batch)
            if next_offset is None:
                break
            offset = next_offset

        docs = []
        for p in all_points:
            payload = p.payload or {}
            updated_raw = payload.get("updated_at")
            docs.append({
                "doc_id":     payload.get("doc_id", ""),
                "text":       payload.get("text", ""),
                "embedding":  list(p.vector) if p.vector else [],
                "updated_at": datetime.fromisoformat(updated_raw) if updated_raw else None,
                "metadata":   payload.get("metadata", {}),
            })
        return docs

    # ── Similarity search ───────────────────────────────────────────────
    def search(
        self,
        collection: str,
        query_vector: List[float],
        top_k: int = 3,
    ) -> List[Dict]:
        """
        Return the top-k most similar documents using Qdrant's built-in
        cosine similarity search.
        """
        response = self._client.query_points(
            collection_name=collection,
            query=query_vector,
            limit=top_k,
            with_payload=True,
        )
        hits = response.points
        results = []
        for hit in hits:
            payload = hit.payload or {}
            updated_raw = payload.get("updated_at")
            results.append({
                "doc_id":     payload.get("doc_id", ""),
                "text":       payload.get("text", ""),
                "updated_at": datetime.fromisoformat(updated_raw) if updated_raw else None,
                "metadata":   payload.get("metadata", {}),
                "_score":     hit.score,
            })
        return results
        # hits = self._client.search(
        #     collection_name=collection,
        #     query_vector=query_vector,
        #     limit=top_k,
        #     with_payload=True,
        # )
        # results = []
        # for hit in hits:
        #     payload = hit.payload or {}
        #     updated_raw = payload.get("updated_at")
        #     results.append({
        #         "doc_id":     payload.get("doc_id", ""),
        #         "text":       payload.get("text", ""),
        #         "updated_at": datetime.fromisoformat(updated_raw) if updated_raw else None,
        #         "metadata":   payload.get("metadata", {}),
        #         "_score":     hit.score,
        #     })
        # return results

    # ── VectorDBAdapter polling interface ───────────────────────────────

    def describe_collection(self, collection: str) -> Dict:
        """Liveness probe used by VectorDBAdapter.connect()."""
        info = self._client.get_collection(collection)
        return {
            "collection": collection,
            "count":      info.points_count,
            "status":     str(info.status),
        }

    def fetch_updated(
        self,
        collection: str,
        since=None,
        timestamp_field: str = "updated_at",
    ) -> List[Dict]:
        """
        Return docs newer than `since` — used by VectorDBAdapter.poll().
        Qdrant does not natively filter by datetime ranges on payload fields
        without a numeric index, so we scroll all and filter in Python.
        For large collections, add a numeric payload index on `updated_at_ts`.
        """
        all_docs = self.list_all(collection)
        if since is None:
            return all_docs
        return [
            d for d in all_docs
            if d.get(timestamp_field) and d[timestamp_field] > since
        ]


# ── Connect and verify ───────────────────────────────────────────────
vector_db = QdrantVectorDB(host=QDRANT_HOST, port=QDRANT_PORT)

# Quick health-check — will raise if Qdrant container is not running
collections = vector_db._client.get_collections().collections
print(f"✅ Qdrant connected — {len(collections)} existing collection(s)")
for c in collections:
    print(f"   • {c.name}")

✅ Qdrant connected — 0 existing collection(s)


## 3. Vectorizer + Upload Documents to Qdrant

Edit `DOCUMENTS` below to add your own text entries.  
Each entry is a `(doc_id, text)` tuple — `doc_id` becomes the `source_id` tag in Redis.

In [87]:
class STVectorizer:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.dims  = self.model.get_sentence_embedding_dimension()

    def embed(self, text: str) -> List[float]:
        return self.model.encode(text).tolist()

    def embed_many(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts).tolist()


vectorizer = STVectorizer()
print(f"Vectorizer ready  dims={vectorizer.dims}")

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1475.00it/s]


Vectorizer ready  dims=384


In [49]:
# ── Define your documents here ───────────────────────────────────────
# Each tuple: (doc_id, text_content)
# doc_id becomes the source_id tag that links the Vector DB entry to
# cached responses in Redis — changing the doc invalidates those entries.

COLLECTION = "knowledge_base"

DOCUMENTS = [
    (
        "doc:climate",
        "Climate change is primarily driven by human emissions of greenhouse gases "
        "like CO2 from fossil fuels, deforestation, and industrial processes. "
        "These gases trap heat in the atmosphere, raising global temperatures."
        "Climate change is also reflected in a wide range of measurable environmental indicators beyond greenhouse gas concentrations. Global average surface temperatures have risen significantly since the late 19th century, with the most rapid warming occurring in recent decades. This warming is closely linked to the increased frequency and intensity of extreme weather events such as heatwaves, heavy rainfall, droughts, and tropical storms. Ocean temperatures have also increased, contributing to coral bleaching and the disruption of marine ecosystems, while thermal expansion and melting glaciers and ice sheets have driven a steady rise in sea levels. Arctic sea ice extent has declined sharply, particularly during summer months, reducing the Earth's albedo effect and accelerating warming. Additionally, climate data shows shifts in precipitation patterns, with some regions experiencing more intense rainfall and flooding, while others face prolonged dry periods. Atmospheric CO₂ concentrations have surpassed 420 parts per million in recent years—levels not seen in millions of years—highlighting the unprecedented rate of change. Together, these datasets provide strong, consistent evidence of a warming planet and ongoing climate system disruption.",
    ),
    (
        "doc:python",
        "Python is a high-level, interpreted programming language known for its "
        "clean syntax. It is widely used for web development, data science, "
        "machine learning, automation, and scripting."
        "Python is a high-level, interpreted programming language widely used across many domains due to its readability and versatility. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming, making it suitable for both beginners and experienced developers. Python’s extensive standard library and ecosystem of third-party packages enable applications in web development, data analysis, artificial intelligence, scientific computing, automation, and more. Popular frameworks like Django and Flask are used for building web applications, while libraries such as NumPy, pandas, and TensorFlow support data science and machine learning tasks. Python uses dynamic typing and automatic memory management, which simplifies development and reduces the need for boilerplate code. Its large global community contributes to continuous improvements, abundant documentation, and a wide range of open-source tools, making Python one of the most in-demand and widely adopted programming languages today.",
    ),
    (
        "doc:redis",
        "Redis is an open-source, in-memory data structure store used as a "
        "database, cache, and message broker. It supports strings, hashes, "
        "lists, sets, sorted sets, bitmaps, and more."
        "Redis is an open-source, in-memory data structure store commonly used as a database, cache, and message broker due to its high performance and low latency. Unlike traditional disk-based databases, Redis stores data primarily in RAM, allowing it to process millions of requests per second with sub-millisecond response times. It supports a variety of data structures such as strings, hashes, lists, sets, sorted sets, bitmaps, and hyperloglogs, making it highly flexible for different use cases. Redis also provides features like persistence (RDB snapshots and AOF logs), replication, and clustering for high availability and scalability. It is frequently used for caching frequently accessed data, session storage in web applications, real-time analytics, pub/sub messaging systems, and queue management. Additionally, Redis includes built-in support for transactions, Lua scripting, and expiration of keys, which helps developers build efficient and responsive systems.",
    ),
    (
        "doc:transformers",
        "Transformer neural networks use a self-attention mechanism to process "
        "sequences in parallel. They underpin modern LLMs like GPT and BERT, "
        "excelling at tasks such as translation, summarisation, and question answering."
        "Transformers are a class of deep learning models that have become the foundation of modern natural language processing and many other AI applications. Introduced in the 2017 paper “Attention Is All You Need,” transformers rely on a mechanism called self-attention to process input data, allowing them to weigh the importance of different words or tokens in a sequence regardless of their position. This enables them to capture long-range dependencies more effectively than earlier models like recurrent or convolutional neural networks. Transformers consist of encoder and decoder layers, each made up of attention heads and feedforward neural networks, which work together to generate contextual representations of data. They are highly parallelizable, making them efficient to train on large datasets using modern hardware like GPUs and TPUs. Popular transformer-based models include BERT, GPT, and T5, which are used for tasks such as text classification, translation, summarization, question answering, and code generation. Beyond text, transformers have also been adapted for use in computer vision, speech processing, and multimodal systems, demonstrating their flexibility and broad impact across artificial intelligence.",
    ),
    (
        "doc:rag",
        "Retrieval-Augmented Generation (RAG) enhances LLM responses by retrieving "
        "relevant documents from a knowledge base at query time and injecting them "
        "into the prompt context before generation."
        "Retrieval-Augmented Generation (RAG) is a hybrid approach in artificial intelligence that combines information retrieval with text generation to produce more accurate and context-aware outputs. Instead of relying solely on a model’s internal knowledge, RAG systems first retrieve relevant documents or data from external sources—such as databases, knowledge bases, or vector stores—and then use that information to guide the generation process. This significantly improves factual accuracy, reduces hallucinations, and allows models to access up-to-date or domain-specific knowledge without retraining. A typical RAG pipeline involves embedding user queries, searching for similar content using techniques like vector similarity, and feeding the retrieved context into a generative model (often a transformer-based model) to produce a final response. RAG is widely used in applications like question answering systems, enterprise search, chatbots, and document summarization. It also enables better transparency, since the retrieved sources can be inspected, and offers scalability by allowing systems to update knowledge simply by modifying the underlying data store rather than retraining the entire model.",
    ),
]

# ── Embed and upload ─────────────────────────────────────────────────
print(f"Uploading {len(DOCUMENTS)} documents to collection '{COLLECTION}'…\n")

texts = [text for _, text in DOCUMENTS]
embeddings = vectorizer.embed_many(texts)

for (doc_id, text), embedding in zip(DOCUMENTS, embeddings):
    vector_db.upsert(
        COLLECTION,
        {
            "doc_id":    doc_id,
            "text":      text,
            "embedding": embedding,
            "metadata":  {},
        },
    )
    print(f"  ✅ [{doc_id}]  {text[:70]}…")

print(f"\nTotal docs in '{COLLECTION}': {len(vector_db.list_all(COLLECTION))}")

Uploading 5 documents to collection 'knowledge_base'…

  Created Qdrant collection 'knowledge_base' (dim=384)
  ✅ [doc:climate]  Climate change is primarily driven by human emissions of greenhouse ga…
  ✅ [doc:python]  Python is a high-level, interpreted programming language known for its…
  ✅ [doc:redis]  Redis is an open-source, in-memory data structure store used as a data…
  ✅ [doc:transformers]  Transformer neural networks use a self-attention mechanism to process …
  ✅ [doc:rag]  Retrieval-Augmented Generation (RAG) enhances LLM responses by retriev…

Total docs in 'knowledge_base': 5


## 4. Initialise GliaManager, CacheInvalidator & Groq

In [50]:
manager = GliaManager(
    vectorizer=vectorizer,
    redis_url=REDIS_URL,
    index_name=INDEX_NAME,
    distance_threshold=0.25,
    vector_dims=vectorizer.dims,
)

invalidator  = CacheInvalidator(cache_manager=manager)
groq_client  = Groq(api_key=GROQ_API_KEY)

print("✅ GliaManager ready")
print("✅ CacheInvalidator ready")
print("✅ Groq client ready")

✅ GliaManager ready
✅ CacheInvalidator ready
✅ Groq client ready


## 5. Ask Questions — Cache Hit / Miss Indicator

The `ask()` helper:
1. Searches **Qdrant** for the most relevant document.
2. Checks the **Redis semantic cache** — prints a coloured `[CACHE HIT]` or `[CACHE MISS]` banner.
3. On a miss: calls **Groq**, then stores the response in Redis tagged with the `doc_id`.

Semantically similar follow-up questions will hit the cache without calling Groq again.

In [51]:
def _banner(label: str, color_code: str, width: int = 62) -> str:
    bar = "─" * width
    return f"\033[{color_code}m┌{bar}┐\n│  {label:<{width-2}}│\n└{bar}┘\033[0m"


def ask(
    prompt: str,
    top_k: int = 1,
    verbose: bool = True,
) -> str:
    """
    1. Embed the prompt and search Qdrant for the most relevant doc.
    2. Check the Redis semantic cache.
       → HIT : return cached response immediately (no LLM call).
       → MISS: call Groq, store result tagged with the retrieved doc_id.
    """
    if verbose:
        print(f"\n{'═'*64}")
        print(f"  QUERY: {prompt}")
        print(f"{'═'*64}")

    # ── Step 1: retrieve most-relevant doc from Qdrant ───────────────
    query_vec = vectorizer.embed(prompt)
    results   = vector_db.search(COLLECTION, query_vec, top_k=top_k)

    source_id = None
    context   = ""
    if results:
        best      = results[0]
        source_id = best["doc_id"]
        context   = best["text"]
        if verbose:
            print(f"  Qdrant   → {source_id}  (score={best['_score']:.3f})")
            print(f"  Context  : {context[:80]}…")
    else:
        if verbose:
            print("  Qdrant   → no results found")

    # ── Step 2: check Redis semantic cache ───────────────────────────
    cached = manager.check(prompt)

    if cached is not None:
        if verbose:
            print()
            print(_banner("  [CACHE HIT] — returning stored response (no LLM call)", "92"))
            print(f"\n  Response:\n{cached}\n")
        return cached

    # ── Step 3: cache miss → call Groq ───────────────────────────────
    if verbose:
        print()
        print(_banner("  [CACHE MISS] — calling Groq LLM…", "91"))

    augmented_prompt = prompt
    if context:
        augmented_prompt = (
            f"Use the following context to answer the question.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {prompt}"
        )

    t0 = time.time()
    completion = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": augmented_prompt}],
        max_tokens=256,
    )
    elapsed  = time.time() - t0
    response = completion.choices[0].message.content

    if verbose:
        print(f"  Groq responded in {elapsed:.2f}s")

    # ── Step 4: store in Redis cache ─────────────────────────────────
    sid = source_id or "general"
    manager.store(prompt=prompt, response=response, source_id=sid)

    if verbose:
        print(f"  Stored in cache under source_id='{sid}'")
        print(f"\n  Response:\n{response}\n")

    return response


print(" ask() helper ready")

 ask() helper ready


In [52]:
%%time
# ── First call → CACHE MISS (Groq is called) ─────────────────────────
_ = ask("What causes climate change, does Global warming contribute to it??")


════════════════════════════════════════════════════════════════
  QUERY: What causes climate change, does Global warming contribute to it??
════════════════════════════════════════════════════════════════
  Qdrant   → doc:climate  (score=0.652)
  Context  : Climate change is primarily driven by human emissions of greenhouse gases like C…

┌──────────────────────────────────────────────────────────────┐
│    [CACHE MISS] — calling Groq LLM…                          │
└──────────────────────────────────────────────────────────────┘
  Groq responded in 1.85s
  Stored in cache under source_id='doc:climate'

  Response:
**Causes of climate change**

The rapid changes we’re seeing in the Earth’s climate are mainly driven by **human activities that add greenhouse gases (GHGs) to the atmosphere**. The key sources are:

| Source | Typical GHG | How it heats the planet |
|--------|-------------|-------------------------|
| Burning fossil fuels (coal, oil, natural gas) |

CPU times: user 890 ms

In [53]:
%%time
# ── Semantically similar → CACHE HIT ─────────────────────────────────
_ = ask("What are the main drivers of global warming?")


════════════════════════════════════════════════════════════════
  QUERY: What are the main drivers of global warming?
════════════════════════════════════════════════════════════════
  Qdrant   → doc:climate  (score=0.609)
  Context  : Climate change is primarily driven by human emissions of greenhouse gases like C…

┌──────────────────────────────────────────────────────────────┐
│    [CACHE HIT] — returning stored response (no LLM call)     │
└──────────────────────────────────────────────────────────────┘

  Response:
**Causes of climate change**

The rapid changes we’re seeing in the Earth’s climate are mainly driven by **human activities that add greenhouse gases (GHGs) to the atmosphere**. The key sources are:

| Source | Typical GHG | How it heats the planet |
|--------|-------------|-------------------------|
| Burning fossil fuels (coal, oil, natural gas) |

CPU times: user 284 ms, sys: 0 ns, total: 284 ms
Wall time: 46.3 ms


In [54]:
%%time
# ── Another topic — first call = MISS ────────────────────────────────
_ = ask("Why is Python a popular programming language?")


════════════════════════════════════════════════════════════════
  QUERY: Why is Python a popular programming language?
════════════════════════════════════════════════════════════════
  Qdrant   → doc:python  (score=0.685)
  Context  : Python is a high-level, interpreted programming language known for its clean syn…

┌──────────────────────────────────────────────────────────────┐
│    [CACHE MISS] — calling Groq LLM…                          │
└──────────────────────────────────────────────────────────────┘
  Groq responded in 0.72s
  Stored in cache under source_id='doc:python'

  Response:
Python is popular for several reasons that all come from the characteristics highlighted in the context:

* **Readable, clean syntax** – Python’s high‑level, interpreted nature makes the code easy to write and read, which appeals to both beginners and experienced developers.

* **Versatility across domains** – It works well for web development, data science, machine learning, automation, scripti

In [55]:
%%time
# ── Repeat (near-identical) → HIT ────────────────────────────────────
_ = ask("Why do developers love Python?")


════════════════════════════════════════════════════════════════
  QUERY: Why do developers love Python?
════════════════════════════════════════════════════════════════
  Qdrant   → doc:python  (score=0.680)
  Context  : Python is a high-level, interpreted programming language known for its clean syn…

┌──────────────────────────────────────────────────────────────┐
│    [CACHE HIT] — returning stored response (no LLM call)     │
└──────────────────────────────────────────────────────────────┘

  Response:
Python is popular for several reasons that all come from the characteristics highlighted in the context:

* **Readable, clean syntax** – Python’s high‑level, interpreted nature makes the code easy to write and read, which appeals to both beginners and experienced developers.

* **Versatility across domains** – It works well for web development, data science, machine learning, automation, scripting, and scientific computing, thanks to its broad range of built‑in and third‑party libr

In [56]:
%%time
# ── Novel topic → MISS, then stored ──────────────────────────────────
_ = ask("How do transformer neural networks work?")


════════════════════════════════════════════════════════════════
  QUERY: How do transformer neural networks work?
════════════════════════════════════════════════════════════════
  Qdrant   → doc:transformers  (score=0.612)
  Context  : Transformer neural networks use a self-attention mechanism to process sequences …

┌──────────────────────────────────────────────────────────────┐
│    [CACHE MISS] — calling Groq LLM…                          │
└──────────────────────────────────────────────────────────────┘
  Groq responded in 0.66s
  Stored in cache under source_id='doc:transformers'

  Response:
**In a nutshell, a transformer is a neural network that processes an entire sequence at once, using *self‑attention* to learn which parts of the sequence matter most to each token.**

---

### 1.  Core idea – *self‑attention*

* **Query, Key, Value** – For every token in the input, the model creates three vectors by multiplying the token’s embedding by learned weight matrices.  
  * **Que

## 6. Update a Document in Qdrant → Invalidate Cache

When source content changes the cached responses are stale.  
This section shows how to:
1. Update the document text in Qdrant (upsert with same `doc_id`).
2. Call `CacheInvalidator.delete_by_tag(doc_id)` to remove stale cache entries.
3. Re-ask the same question — now a **MISS** → fresh Groq response stored.

In [57]:
# ── Show the document BEFORE update ─────────────────────────────────
TARGET_DOC = "doc:climate"

doc = vector_db.get(COLLECTION, TARGET_DOC)
print(f"Document  : {TARGET_DOC}")
print(f"Updated at: {doc['updated_at']}")
print(f"\nText (before):\n{doc['text']}")

Document  : doc:climate
Updated at: 2026-05-06 08:18:16.918563

Text (before):
Climate change is primarily driven by human emissions of greenhouse gases like CO2 from fossil fuels, deforestation, and industrial processes. These gases trap heat in the atmosphere, raising global temperatures.Climate change is also reflected in a wide range of measurable environmental indicators beyond greenhouse gas concentrations. Global average surface temperatures have risen significantly since the late 19th century, with the most rapid warming occurring in recent decades. This warming is closely linked to the increased frequency and intensity of extreme weather events such as heatwaves, heavy rainfall, droughts, and tropical storms. Ocean temperatures have also increased, contributing to coral bleaching and the disruption of marine ecosystems, while thermal expansion and melting glaciers and ice sheets have driven a steady rise in sea levels. Arctic sea ice extent has declined sharply, particularly d

In [79]:
# ── Update the document in Qdrant ────────────────────────────────────
UPDATED_TEXT = (
    "Climate change is caused by both natural factors and human activities. "
    "The dominant driver since the Industrial Revolution is the burning of "
    "fossil fuels (coal, oil, natural gas), releasing CO2 and methane. "
    "Deforestation, agriculture, and cement production are secondary contributors. "
    "The resulting greenhouse effect raises average global temperatures, "
    "intensifying extreme weather events and sea-level rise."
    "Global warming is essentially the Earth’s way of telling us the thermostat is stuck."
    "Driven primarily by the thickening blanket of greenhouse gases—like carbon dioxide and methane—it traps solar heat that would otherwise escape into space, steadily cranking up the planet's average surface temperature."
    "This fever is the primary engine behind climate change, a broader disruption of the planet's atmospheric and oceanic systems."
    "As the atmosphere warms, it gains the energy to hold more moisture and shift wind patterns, leading to a feast or famine weather cycle: more intense, destructive storms and flooding in some regions, while others suffer through prolonged, record-breaking droughts."
    "From the melting of polar ice caps to the rising sea levels that threaten coastal geography, global warming acts as a chaotic force multiplier, turning predictable seasonal rhythms into a series of increasingly erratic and extreme environmental shifts."
)

new_embedding = vectorizer.embed(UPDATED_TEXT)

vector_db.upsert(
    COLLECTION,
    {
        "doc_id":    TARGET_DOC,
        "text":      UPDATED_TEXT,
        "embedding": new_embedding,
        "metadata":  {"version": 2},
    },
)

updated = vector_db.get(COLLECTION, TARGET_DOC)
print(f"✅ Qdrant updated — new timestamp: {updated['updated_at']}")
print(f"\nNew text:\n{UPDATED_TEXT}")

  Created Qdrant collection 'knowledge_base' (dim=384)
✅ Qdrant updated — new timestamp: 2026-05-06 08:31:48.005521

New text:
Climate change is caused by both natural factors and human activities. The dominant driver since the Industrial Revolution is the burning of fossil fuels (coal, oil, natural gas), releasing CO2 and methane. Deforestation, agriculture, and cement production are secondary contributors. The resulting greenhouse effect raises average global temperatures, intensifying extreme weather events and sea-level rise.Global warming is essentially the Earth’s way of telling us the thermostat is stuck.Driven primarily by the thickening blanket of greenhouse gases—like carbon dioxide and methane—it traps solar heat that would otherwise escape into space, steadily cranking up the planet's average surface temperature.This fever is the primary engine behind climate change, a broader disruption of the planet's atmospheric and oceanic systems.As the atmosphere warms, it gains the e

In [80]:
# ── Invalidate stale cache entries for this document ─────────────────
print(f"Invalidating cache for source_id='{TARGET_DOC}'…")

deleted = invalidator.delete_by_tag(TARGET_DOC)

if deleted > 0:
    print(_banner(f"🗑️  INVALIDATED {deleted} cache entry/entries for '{TARGET_DOC}'", "93"))
else:
    print(f"  No cache entries found for '{TARGET_DOC}' (already empty).")

Invalidating cache for source_id='doc:climate'…
  No cache entries found for 'doc:climate' (already empty).


In [81]:
%%time
# ── Re-ask — cache is gone → fresh Groq call with updated context ────
_ = ask("What causes climate change?")


════════════════════════════════════════════════════════════════
  QUERY: What causes climate change?  How does globakl warming contribute to it?
════════════════════════════════════════════════════════════════
  Qdrant   → doc:climate  (score=0.706)
  Context  : Climate change is caused by both natural factors and human activities. The domin…

┌──────────────────────────────────────────────────────────────┐
│    [CACHE MISS] — calling Groq LLM…                          │
└──────────────────────────────────────────────────────────────┘
  Groq responded in 1.65s
  Stored in cache under source_id='doc:climate'

  Response:
**What causes climate change?**  
- **Natural drivers** (e.g., volcanic eruptions, solar output variations, orbital changes) are part of the Earth’s long‑term climate cycle.  
- **Human activities** dominate the recent trend.  The burning of fossil fuels (coal, oil, natural gas) releases large amounts of CO₂ and methane, the most potent greenhouse gases.  D

CPU times

In [82]:
%%time
_ = ask("What are the main drivers of global warming?")


════════════════════════════════════════════════════════════════
  QUERY: What are the main drivers of global warming?
════════════════════════════════════════════════════════════════
  Qdrant   → doc:climate  (score=0.673)
  Context  : Climate change is caused by both natural factors and human activities. The domin…

┌──────────────────────────────────────────────────────────────┐
│    [CACHE MISS] — calling Groq LLM…                          │
└──────────────────────────────────────────────────────────────┘
  Groq responded in 0.60s
  Stored in cache under source_id='doc:climate'

  Response:
**Main drivers of global warming**

1. **Burning of fossil fuels** – coal, oil, and natural gas are the largest source of atmospheric CO₂ and methane, the primary greenhouse gases.  
2. **Deforestation and land‑use change** – removal of trees reduces carbon sequestration and adds CO₂ to the air.  
3. **Agriculture** – livestock, rice paddies, and other farming practices emit methane and nitrous

## 7. Inspect Redis Cache Contents

Browse every entry currently stored in the Redis index —  
prompt, source_id tag, and similarity-vector dimension count.

In [62]:
import redis as _redis_lib

r = _redis_lib.from_url(REDIS_URL, decode_responses=False)

pattern = f"{INDEX_NAME}:entry:*"
keys    = r.keys(pattern)

print(f"Redis index  : '{INDEX_NAME}'")
print(f"Cache entries: {len(keys)}")
print()

if not keys:
    print("  (empty — run cells in Section 5 first)")
else:
    for i, key in enumerate(sorted(keys), 1):
        key_str = key.decode() if isinstance(key, bytes) else key
        try:
            doc = r.json().get(key_str, "$")
            if doc and len(doc) > 0:
                entry    = doc[0]
                prompt   = entry.get("prompt", "")[:80]
                source   = entry.get("source_id", "—")
                vec_len  = len(entry.get("prompt_vector", []))
                response = entry.get("response", "")[:60]
                print(f"  [{i:02d}] key       : {key_str}")
                print(f"       source_id : {source}")
                print(f"       prompt    : {prompt}…")
                print(f"       response  : {response}…")
                print(f"       vector dim: {vec_len}")
                print()
        except Exception as e:
            print(f"  [{i:02d}] {key_str} — could not read ({e})")

Redis index  : 'rag_cache'
Cache entries: 4

  [01] key       : rag_cache:entry:93c062ab5717fab58512e3a7a63e211237c6c17c4d4799d188fd467b5527d2eb
       source_id : doc:transformers
       prompt    : How do transformer neural networks work?…
       response  : **In a nutshell, a transformer is a neural network that proc…
       vector dim: 384

  [02] key       : rag_cache:entry:9c2400054219022b9d0ce951c6cb6e5ba3947c5790ec79ee76bc49ad243530e3
       source_id : doc:python
       prompt    : Why is Python a popular programming language?…
       response  : Python is popular for several reasons that all come from the…
       vector dim: 384

  [03] key       : rag_cache:entry:b90030adfc948e6c709e5b460ab0111beebb3a7732d9e92b80cfd12a7f5cf55a
       source_id : doc:climate
       prompt    : What causes climate change?…
       response  : Climate change is driven mainly by human activities that inc…
       vector dim: 384

  [04] key       : rag_cache:entry:fd457e766ea26857161f3250281bd255b

In [63]:
# ── Show a summary grouped by source_id ─────────────────────────────
from collections import defaultdict

by_source = defaultdict(list)

for key in keys:
    key_str = key.decode() if isinstance(key, bytes) else key
    try:
        doc = r.json().get(key_str, "$")
        if doc and len(doc) > 0:
            entry  = doc[0]
            source = entry.get("source_id", "unknown")
            prompt = entry.get("prompt", "")[:60]
            by_source[source].append(prompt)
    except Exception:
        pass

print(f"Cache entries grouped by source_id ({len(by_source)} sources):\n")
for source, prompts in sorted(by_source.items()):
    print(f"  {source}  ({len(prompts)} entr{'y' if len(prompts)==1 else 'ies'})")
    for p in prompts:
        print(f"    • {p}…")
    print()

Cache entries grouped by source_id (3 sources):

  doc:climate  (2 entries)
    • What causes climate change?…
    • What are the main drivers of global warming?…

  doc:python  (1 entry)
    • Why is Python a popular programming language?…

  doc:transformers  (1 entry)
    • How do transformer neural networks work?…



In [64]:
# ── Show current state of Qdrant ────────────────────────────────────
all_docs = vector_db.list_all(COLLECTION)

print(f"Qdrant collection '{COLLECTION}' — {len(all_docs)} documents:\n")
for doc in all_docs:
    meta = doc.get("metadata") or {}
    print(f"  doc_id    : {doc['doc_id']}")
    print(f"  updated_at: {doc['updated_at']}")
    print(f"  metadata  : {meta}")
    print(f"  text      : {doc['text'][:80]}…")
    print(f"  embedding : [{doc['embedding'][0]:.4f}, {doc['embedding'][1]:.4f}, … ] dim={len(doc['embedding'])}")
    print()

Qdrant collection 'knowledge_base' — 5 documents:

  doc_id    : doc:python
  updated_at: 2026-05-06 08:18:16.934378
  metadata  : {}
  text      : Python is a high-level, interpreted programming language known for its clean syn…
  embedding : [-0.0543, -0.0353, … ] dim=384

  doc_id    : doc:climate
  updated_at: 2026-05-06 08:20:32.299472
  metadata  : {'version': 2}
  text      : Climate change is caused by both natural factors and human activities. The domin…
  embedding : [0.0004, 0.0368, … ] dim=384

  doc_id    : doc:rag
  updated_at: 2026-05-06 08:18:16.988438
  metadata  : {}
  text      : Retrieval-Augmented Generation (RAG) enhances LLM responses by retrieving releva…
  embedding : [-0.0890, 0.0028, … ] dim=384

  doc_id    : doc:transformers
  updated_at: 2026-05-06 08:18:16.970831
  metadata  : {}
  text      : Transformer neural networks use a self-attention mechanism to process sequences …
  embedding : [-0.1504, -0.0420, … ] dim=384

  doc_id    : doc:redis
  updated_at

## 8. Full Cleanup

Drops the RediSearch index, deletes all cached documents, and deletes the Qdrant collection.  
Run this cell when you are done testing.

In [83]:
# Drop Redis index + all documents
manager.delete_index(drop_documents=True)
print("✅ Redis index and all cached documents deleted.")

# Delete the Qdrant collection
vector_db._client.delete_collection(COLLECTION)
print(f"✅ Qdrant collection '{COLLECTION}' deleted.")
print("\nAll done. Re-run from Section 3 to start fresh.")

✅ Redis index and all cached documents deleted.
✅ Qdrant collection 'knowledge_base' deleted.

All done. Re-run from Section 3 to start fresh.
